# 0. Imports

In [1]:
# Libraries
import os
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

from datetime import date
from dataclasses import dataclass, field
from dateutil.relativedelta import relativedelta, WE
from scipy.optimize import brentq

In [2]:
# Global variables
DIR_PATH = os.getcwd()
DATA_PATH = os.path.join(DIR_PATH, 'market_data_split.csv')

t0 = dt.datetime(2026, 8, 19)

In [101]:
# Data
df = pd.read_csv(DATA_PATH, parse_dates=['maturity']) # SOFR quotes
display(df)

,type,tenor,quote,maturity
0,SOFR,ON,3.620,2026-08-20
1,SR1,Aug-26,96.361,2026-09-01
2,SR1,Sep-26,96.301,2026-10-01
3,SR3,Sep-26,96.195,2026-12-16
4,SR3,Dec-26,96.072,2027-03-17
5,SR3,Mar-27,96.015,2027-06-16
6,SR3,Jun-27,95.999,2027-09-15
7,OIS,2Y,4.000,2028-08-21
8,OIS,3Y,4.000,2029-08-20
9,OIS,5Y,4.040,2031-08-19


# 1. Bootstrapping

## 1.1 Discount curve with exact method

In [4]:
# Date functions

def act360(d1, d2):
    return (d2 - d1).days/360

def act365(d1, d2):
    return (d2 - d1).days/365

def following(d):
    d = np.datetime64(d, 'D')
    return np.busday_offset(d, offsets=0 ,roll='following')

def annual_schedule(t0, n_years):
    delta = relativedelta(years = n_years)
    T = t0 + delta
    return following(T)

In [5]:
# Check sanity of annual_schedule

ois = df[df['type'] == 'OIS']
ois_years = ois["tenor"].str.rstrip("Y").astype(int) 

for n, expected in zip(ois_years, ois["maturity"]):
    sched = annual_schedule(t0, n)
    print(sched)
    

2028-08-21
2029-08-20
2031-08-19
2033-08-19
2036-08-19
2041-08-19
2046-08-20
2056-08-21


In [124]:
from typing import Any


@dataclass
class Instrument:
    type: str
    tenor: str
    quote: float
    start: date
    end: date

def build_instrument(df, t0) -> list[Instrument]:

    insts = []

    for _, row in df.iterrows():

        if row.type == 'SOFR':
            inst = Instrument(row.type, row.tenor, row.quote, t0, row.maturity)

        elif row.type == 'SR1':
            delta = relativedelta(months = -1)
            start = row.maturity + delta
            if start < t0:

                start = t0
            inst = Instrument(row.type, row.tenor, row.quote, start, row.maturity)

        elif row.type == 'SR3':
            delta = relativedelta(months = -3, day = 1, weekday=WE(+3))
            start = row.maturity + delta
            inst = Instrument(row.type, row.tenor, row.quote, start, row.maturity)

        elif row.type == 'OIS':
            tenor_strip = int(row.tenor.rstrip('Y'))
            # Build pay dates
            pay_dates = []
            for year in range(tenor_strip):
                pay_dates.append(annual_schedule(t0, year + 1))
            inst = OIS(row.type, row.tenor, row.quote, t0, row.maturity, 
                       pay_dates=pay_dates
                       )

        insts.append(inst)

    return insts

@dataclass
class OIS(Instrument):
    pay_dates: list[date]

class Curve:
    def __init__(self, t0, t=None, logP=None):
        self.t0 = t0
        self.t = np.array([0.0]) if t is None else t # Time to maturity
        self.logP = np.array([0.0]) if logP is None else logP

    def _t(self, d):
        # Get the time to maturity in ACT/365
        d = np.asarray(d, dtype='datetime64[D]')
        t0 = np.datetime64(self.t0, 'D')
        return (d - t0).astype(float) / 365.0

    def P(self, d):
        tq = self._t(d)
        # Check feasible date
        if np.any(tq < 0) or np.any(tq > self.t[-1] + 1e-12):
            print(tq, self.t[-1])
            raise ValueError("Date is not in range")
        # Interpolation if within range
        return np.exp(np.interp(tq, self.t, self.logP))


    def with_node(self, d, P):
        tn = self._t(d)
        return Curve(self.t0, np.append(self.t, tn), np.append(self.logP, np.log(P)))

    def fwd_simple(self, s, e):
        # Get the delta in ACT /360
        delta = act360(s, e)
        # Compute the forward
        return 1/delta * (self.P(s)/self.P(e) - 1)


def model_quote(inst, curve):
    type = inst.type
    start = inst.start
    end = inst.end

    if type == 'SOFR':
        quote = curve.fwd_simple(start, end)
        
    elif type in ('SR1', 'SR3'):
        quote = (1 - curve.fwd_simple(start, end))

    elif type == 'OIS':
        s = 0
        payDates = list(inst.pay_dates)
        payDates.insert(0, inst.start)
        for i in range(1, len(payDates)):
            delta = act360(pd.Timestamp(payDates[i - 1]), pd.Timestamp(payDates[i]))
            P = curve.P(payDates[i])
            s += delta * P
        quote = (1 - P) / s

    else:
        raise ValueError("Instrument type not recognized")

    return 100 * quote

In [127]:
def residual(p, inst, curve):
        curve = curve.with_node(inst.end, p)
        diff = model_quote(inst, curve) - inst.quote
        return diff

insts = build_instrument(df, t0)
curve = Curve(t0) 
ends = [inst.end for inst in insts]
tmp = pd.DataFrame([insts, ends], ['INST', 'END']).T.sort_values('END')
res = []

for _, row in tmp.iterrows():
    inst = row.INST

    # Solve for p
    p = brentq(residual, 0.1, 1, args = (inst, curve), xtol=1e-14)
    
    # Update the curve
    curve = curve.with_node(inst.end, p)
    # save data
    res.append((inst.end, p))

In [128]:
curve.logP

array([ 0.00000000e+00, -1.00550500e-04, -1.31322068e-03, -4.39097952e-03,
       -1.24243342e-02, -2.23044756e-02, -3.23272735e-02, -4.23901118e-02,
       -7.98440177e-02, -1.19491990e-01, -2.00968598e-01, -2.87433030e-01,
       -4.23978151e-01, -6.58636843e-01, -8.89752797e-01, -1.31885766e+00])

In [105]:
OIS_dates = tmp.iloc[-1]['INST'].pay_dates
OIS = df[df['type'] == 'OIS']

# Dictionnaire associant chaque date OIS à son quote (en s'assurant que les types Timestamp correspondent)
quote_map = dict(zip(pd.to_datetime(OIS['maturity']), OIS['quote']))

rows = []
for elem in OIS_dates:
  dt = pd.Timestamp(elem)
  rows.append({
      'delta': act360(t0, dt),
      'quote': quote_map.get(dt, np.nan),
  })

inter_df = pd.DataFrame(rows)
inter_df.loc[0, 'quote'] = 3.963

inter_df['quote'] = (
    inter_df.set_index('delta')['quote']
    .interpolate(method='values')
    .values
)

# Conversion des quotes en taux décimaux (ex: 3.963 -> 0.03963)
rates = inter_df['quote'].values / 100.0
deltas = inter_df['delta'].values

discount_factors = []
annuity_sum = 0.0  # Représente sum_{i=1}^{n-1} alpha_i * P(0, t_i)

for i in range(len(inter_df)):
  # Calcul du accrual factor alpha_i = delta_i - delta_{i-1} (avec delta_0 = 0)
  prev_delta = 0.0 if i == 0 else deltas[i - 1]
  alpha = deltas[i] - prev_delta

  S = rates[i]

  # Formule explicite du bootstrapping
  P = (1.0 - S * annuity_sum) / (1.0 + S * alpha)

  discount_factors.append(P)
  annuity_sum += alpha * P

inter_df['P'] = discount_factors

inter_df

,delta,quote,P
0,1.013889,3.963000,0.961372
1,2.036111,4.000000,0.923260
2,3.047222,4.000000,0.887371
3,4.058333,4.019973,0.852171
4,5.072222,4.040000,0.817931
5,6.088889,4.080055,0.783621
6,7.102778,4.120000,0.750166
7,8.122222,4.160182,0.717308
8,9.133333,4.200036,0.685489
9,10.147222,4.240000,0.654366
